## __MCDI - SEMANA 3__
### __Datos__:
#### __Grupo:__ 8
#### __Fecha:__ 15/06/2026
#### __Curso:__ MCDI500.202681.2556

In [ ]:
#Importación de Librerías.
import pandas as pd
import time
from abc import ABC, abstractmethod

#Clase Abstracta Base.
class FinancialDataset(ABC):
    def __init__(self, archivo):
        #Proceso de Encapsulamiento.
        self._archivo = archivo
        self._df = None
    @abstractmethod
    def cargar(self):
        pass
    @abstractmethod
    def limpiar(self):
        pass
    @abstractmethod
    def preparar(self):
        pass
    @abstractmethod
    def validar(self):
        pass
    def obtener_dataframe(self):
        return self._df.copy()
    
#Dataset Bitcoin.
class BitcoinDataset(FinancialDataset):
    #Proceso de carga de datos.
    def cargar(self):
        print("Cargando datos...")
        self._df = pd.read_csv(
            self._archivo,
            encoding="utf-8",
            dtype={
                "Open": "float64",
                "High": "float64",
                "Low": "float64",
                "Close": "float64",
                "Adj Close": "float64",
                "Volume": "int64"
            },
            parse_dates=["Date"]
        )
    #Proceso de limpieza de datos.
    def limpiar(self):
        print("Limpiando datos...")
        #Conversión de fecha.
        self._df["Date"] = pd.to_datetime(self._df["Date"])
        #Ordenamiento por fecha.
        self._df = self._df.sort_values("Date")
        #Selección de columnas numéricas.
        columnas_numericas = (self._df.select_dtypes(include="number").columns)
        #Tratamiento de registros nulos.
        for columna in columnas_numericas:
            cantidad_nulos = (self._df[columna].isna().sum())
            if cantidad_nulos > 0:
                promedio = (self._df[columna].mean())  #Cálculo de promedio.
                self._df[columna] = (self._df[columna].fillna(promedio))
                print(
                    f"{columna}: "
                    f"{cantidad_nulos} nulos reemplazados por {promedio:.2f}"
                )
        #Eliminación de duplicados.
        self._df = (self._df.drop_duplicates())
    #Proceso de preparación de datos.    
    def preparar(self):
        print("Preparando datos...")
        self._df = (self._df.set_index("Date"))
        #Cálculo de variación diaria.
        self._df["Return"] = (self._df["Close"].pct_change().fillna(0))
        #Cálculo de diferencia absoluta.
        self._df["Price_Diff"] = (self._df["Close"] - self._df["Open"])
        #Cálculo de volatilidad diaria.
        self._df["Volatility"] = ((self._df["High"] - self._df["Low"]) / self._df["Open"]) * 100
        #Cálculo de media móvil de 7 días.
        self._df["MA_7"] = (self._df["Close"].rolling(window=7,min_periods=1).mean())  #Las posiciones lógicas de 0 a 6 se completan con promedio, para no alterar el análisis (se quita reemplazar por 0).
        #Cálculo de media móvil de 30 días.
        self._df["MA_30"] = (self._df["Close"].rolling(window=30,min_periods=1).mean())  #Las posiciones lógicas de 0 a 29 se completan con promedio, para no alterar el análisis (se quita reemplazar por 0).
        #Cálculo de volumen promedio móvil.
        self._df["Volume_MA30"] = (self._df["Volume"].rolling(window=30,min_periods=1).mean())  #Las posiciones lógicas de 0 a 29 se completan con promedio, para no alterar el análisis (se quita reemplazar por 0).
    #Proceso de validación de datos.
    def validar(self):
        print("\nValidaciones:")
        open_negativo = (self._df["Open"] < 0).sum()  #Valor de Open no puede ser negativo.
        high_menor_low = (self._df["High"] < self._df["Low"]).sum()  #Valor High debe ser mayor o igual que Low.
        volumen_negativo = (self._df["Volume"] < 0).sum()  #Valor de Volume no puede ser negativo.
        print(
            f"Open negativos: {open_negativo}"
        )
        print(
            f"High < Low: {high_menor_low}"
        )
        print(
            f"Volume negativos: {volumen_negativo}"
        )

# Segundo Dataset parar demostrar Polimorfismo (Dataset no Existe, solo es un ejemplo).
class StockDataset(FinancialDataset):
    def cargar(self):
        print("Carga de acciones")
    def limpiar(self):
        print("Limpieza de acciones")
    def preparar(self):
        print("Preparación de acciones")
    def validar(self):
        print("Validación de acciones")
        
#Pipeline.
class DataPipeline:
   def __init__(self, dataset):
       self.dataset = dataset
   def ejecutar(self):
       self.dataset.cargar()
       self.dataset.limpiar()
       self.dataset.preparar()
       self.dataset.validar()
       return self.dataset
   
#Complejidad: Comparación de Algoritmos.
class ComplejidadAnalisis:
    @staticmethod
    def media_movil_pandas(df):
        inicio = time.perf_counter()  #Inicio del contador.
        resultado = (df["Close"].rolling(30).mean())  #Proceso a evaluar.
        fin = time.perf_counter()  #Termino del contador.
        print(
            f"Pandas O(n): {fin - inicio:.6f} segundos."
        )
        return resultado
    @staticmethod
    def media_movil_manual(df):
        inicio = time.perf_counter()  #Inicio del contador.
        resultado = []
        for i in range(len(df)):
            promedio = (df["Close"].iloc[max(0, i - 29): i + 1].mean())  #Proceso a evaluar.
            resultado.append(promedio)
        fin = time.perf_counter()  #Termino del contador.
        print(
            f"Manual O(n²): {fin - inicio:.6f} segundos."
        )
        return resultado
   
#Programa Principal (main).
if __name__ == "__main__":
    bitcoin = BitcoinDataset("bitcoin_dataset.csv")
    pipeline = DataPipeline(bitcoin)
    pipeline.ejecutar()
    df_final = (bitcoin.obtener_dataframe())
    print("\nPrimeros registros:")
    print(df_final.head())  #5 datos por defecto.
    print("\nÚltimos registros:")
    print(df_final.tail())  #5 datos por defecto.
    print("\nComparación de complejidad")
    ComplejidadAnalisis.media_movil_pandas(df_final)
    ComplejidadAnalisis.media_movil_manual(df_final)